# Quantitative Momentum Strategy

"Momentum investing" means investing in the stocks that have increased in price the most.

For this project, we're going to build an investing strategy that selects the 50 stocks with the highest price momentum. From there, we will calculate recommended trades for an equal-weight portfolio of these 50 stocks.


## Library Imports

The first thing we need to do is import the open-source software libraries that we'll be using in this tutorial.

In [2]:
import numpy as np
import pandas as pd
# import requests
import math
from scipy import stats
import xlsxwriter
from datetime import datetime
import yfinance as yf
from statistics import mean

## Importing Our List of Stocks

As before, we'll need to import our list of stocks and our API token before proceeding. Make sure the `.csv` file is still in your working directory and import it with the following command:

In [2]:
stocks = pd.read_csv('/Users/lin/github/quant/sp_500_stocks.csv')

In [3]:
stocks.head(100)

,Ticker
0,A
1,AAL
2,AAP
3,AAPL
4,ABBV
...,...
95,CINF
96,CL
97,CLX
98,CMA


## Making Our First API Call

It's now time to make the first version of our momentum screener!

We need to get one-year price returns for each stock in the universe. Here's how.

In [57]:
symbol = 'CINF'
tickerData = yf.Ticker(symbol)

In [58]:
# Get the historical prices for this ticker
# ticker_df_p1 = tickerData.history(start='2024-07-15', end='2024-07-19')
# ticker_df_p2 = tickerData.history(start='2025-07-15', end='2025-07-19')

In [59]:
ticker_df = tickerData.history(period='1y', end='2025-08-08')

#### Validate `fiftyTwoWeekChangePercent`

In [60]:
ticker_df

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2024-08-08 00:00:00-04:00,123.657626,124.712443,123.599027,124.390137,367400,0.0,0.0
2024-08-09 00:00:00-04:00,124.263165,125.278915,123.501347,125.191002,296700,0.0,0.0
2024-08-12 00:00:00-04:00,125.054279,125.542620,123.843204,123.901802,275400,0.0,0.0
2024-08-13 00:00:00-04:00,124.790569,125.601224,123.696691,125.474251,412000,0.0,0.0
2024-08-14 00:00:00-04:00,125.396124,127.622957,124.683151,127.036957,553900,0.0,0.0
...,...,...,...,...,...,...,...
2025-08-01 00:00:00-04:00,146.789993,146.789993,143.869995,146.190002,694700,0.0,0.0
2025-08-04 00:00:00-04:00,146.520004,149.779999,146.520004,149.740005,523300,0.0,0.0
2025-08-05 00:00:00-04:00,150.389999,151.139999,149.500000,150.429993,569200,0.0,0.0


In [61]:
last_price = ticker_df['Close'].iloc[-1]
first_price = ticker_df['Open'].iloc[0]

In [62]:
last_price

np.float64(151.47000122070312)

In [63]:
first_price

np.float64(123.65762570986202)

In [64]:
price_change = last_price / first_price - 1

In [65]:
price_change

np.float64(1.2249143581011115)

In [66]:
tickerData.info['fiftyTwoWeekChangePercent']

18.169762

## Parsing Our API Call

This API call has all the information we need. We can parse it using the same square-bracket notation as in the first project of this course. Here is an example.

## Executing A Batch API Call & Building Our DataFrame

Just like in our first project, it's now time to execute several batch API calls and add the information we need to our DataFrame.

We'll start by running the following code cell, which contains some code we already built last time that we can re-use for this project. More specifically, it contains a function called `chunks` that we can use to divide our list of securities into groups of 100.

In [6]:
# Function sourced from 
# https://stackoverflow.com/questions/312443/how-do-you-split-a-list-into-evenly-sized-chunks
def chunks(lst, n):
    """Yield successive n-sized chunks from lst."""
    for i in range(0, len(lst), n):
        yield lst[i:i + n]   
        
symbol_groups = list(chunks(stocks['Ticker'], 100))
symbol_strings = []
for i in range(0, len(symbol_groups)):
    symbol_strings.append(' '.join(symbol_groups[i]))

Now we need to create a blank DataFrame and add our data to the data frame one-by-one.

In [5]:
%%time
final_dataframe = pd.DataFrame()

for symbol_string in symbol_strings:
    tickers = yf.Tickers(symbol_string) 
    for symbol in symbol_string.split(' '):
        try:
            return_1year = tickers.tickers[symbol].info['fiftyTwoWeekChangePercent']
            price = tickers.tickers[symbol].info['currentPrice']
        except:
            print(f"{symbol} is skipped")
            pass  
        series = pd.Series([symbol, price, return_1year, 'N/A'])
        final_dataframe = pd.concat([final_dataframe, series.to_frame().T], ignore_index=True)
        
final_dataframe.columns = ['Ticker', 'Price', 'One-Year Price Return', 'Number of Shares to Buy']    

NameError: name 'symbol_strings' is not defined

In [73]:
final_dataframe.head(10)

,Ticker,Price,One-Year Price Return,Number of Shares to Buy
0,A,113.5,-15.738678,N/A
1,AAL,11.03,18.857754,N/A
2,AAP,52.71,-10.448522,N/A
3,AAPL,202.38,-3.2924,N/A
4,ABBV,195.22,5.890644,N/A
5,ABC,195.22,5.890644,N/A
6,ABMD,195.22,5.890644,N/A
7,ABT,127.29,17.17757,N/A
8,ACN,255.37,-18.604576,N/A
9,ADBE,347.8,-31.712872,N/A


## Removing Low-Momentum Stocks

The investment strategy that we're building seeks to identify the 50 highest-momentum stocks in the S&P 500.

Because of this, the next thing we need to do is remove all the stocks in our DataFrame that fall below this momentum threshold. We'll sort the DataFrame by the stocks' one-year price return, and drop all stocks outside the top 50.


In [76]:
final_dataframe.sort_values('One-Year Price Return', ascending=False, inplace=True)

In [77]:
final_dataframe.reset_index(inplace=True)

In [78]:
top50_df = final_dataframe[:50]

In [79]:
top50_df

,index,Ticker,Price,One-Year Price Return,Number of Shares to Buy
0,441,TPR,106.31,188.41562,N/A
1,340,NRG,167.63,143.32996,N/A
2,391,RE,314.25,124.83366,N/A
3,390,RCL,314.25,124.83366,N/A
4,454,UAL,84.57,123.25766,N/A
5,233,HWM,184.26,112.403465,N/A
6,50,AVGO,288.64,103.15316,N/A
7,84,CCL,29.07,101.87499,N/A
8,332,NFLX,1158.6,93.56779,N/A
9,354,ORCL,244.42,91.1622,N/A


## Calculating the Number of Shares to Buy

Just like in the last project, we now need to calculate the number of shares we need to buy. The one change we're going to make is wrapping this functionality inside a function, since we'll be using it again later in this Jupyter Notebook.

Since we've already done most of the work on this, try to complete the following two code cells without watching me do it first!

In [40]:
def portfolio_input():
    global portfolio_size
    portfolio_size = input('Enter the size of your portfolio: ')

    try: 
        print(float(portfolio_size))
        
    except ValueError:
        print('This is not a number! \nplease try again')

In [86]:
portfolio_input()

Enter the size of your portfolio:  50000


50000.0


In [80]:
top50_df.shape[0]

50

In [87]:
position_size = float(portfolio_size)/top50_df.shape[0]

In [88]:
position_size

1000.0

In [93]:
for i in range(top50_df.shape[0]):
    top50_df.loc[i, 'Number of Shares to Buy'] = math.floor(position_size/top50_df.loc[i, 'Price'])

In [94]:
top50_df

,index,Ticker,Price,One-Year Price Return,Number of Shares to Buy
0,441,TPR,106.31,188.41562,9
1,340,NRG,167.63,143.32996,5
2,391,RE,314.25,124.83366,3
3,390,RCL,314.25,124.83366,3
4,454,UAL,84.57,123.25766,11
5,233,HWM,184.26,112.403465,5
6,50,AVGO,288.64,103.15316,3
7,84,CCL,29.07,101.87499,34
8,332,NFLX,1158.6,93.56779,0
9,354,ORCL,244.42,91.1622,4


## Building a Better (and More Realistic) Momentum Strategy

Real-world quantitative investment firms differentiate between "high quality" and "low quality" momentum stocks:

* High-quality momentum stocks show "slow and steady" outperformance over long periods of time
* Low-quality momentum stocks might not show any momentum for a long time, and then surge upwards.

The reason why high-quality momentum stocks are preferred is because low-quality momentum can often be cause by short-term news that is unlikely to be repeated in the future (such as an FDA approval for a biotechnology company).

To identify high-quality momentum, we're going to build a strategy that selects stocks from the highest percentiles of: 

* 1-month price returns
* 3-month price returns
* 6-month price returns
* 1-year price returns

Let's start by building our DataFrame. You'll notice that I use the abbreviation `hqm` often. It stands for `high-quality momentum`.

In [22]:
hqm_columns = [
    'Ticker',
    'Price',
    'Number of Shares to Buy',
    'One-year Price Return',
    'One-year Return Percentile',
    'Six-month Price Return',
    'Six-month Return Percentile',
    'Three-month Price Return',
    'Three-month Return Percentile',
    'One-month Price Return',
    'One-month Return Percentile',
]

hqm_df = pd.DataFrame(columns=hqm_columns)

In [29]:
hqm_df

,0,1,2,3,4,5,6,7,8,9,10
0,A,124.96,N/A,0.886333,N/A,0.978855,N/A,1.070741,N/A,1.043507,N/A
1,AAL,13.25,N/A,1.293945,N/A,0.909403,N/A,1.138316,N/A,1.141258,N/A
2,AAP,60.52,N/A,1.32628,N/A,1.65543,N/A,1.149734,N/A,1.052339,N/A
3,AAPL,232.56,N/A,1.015395,N/A,0.983875,N/A,1.143648,N/A,1.087048,N/A
4,ABBV,207.92,N/A,1.098755,N/A,1.031854,N/A,1.143115,N/A,1.099233,N/A
...,...,...,...,...,...,...,...,...,...,...,...
500,YUM,145.47,N/A,1.08671,N/A,0.948197,N/A,1.015852,N/A,1.006643,N/A
501,ZBH,105.06,N/A,0.922649,N/A,1.007138,N/A,1.140332,N/A,1.098609,N/A
502,ZBRA,322.0,N/A,0.932145,N/A,1.038542,N/A,1.100667,N/A,0.954753,N/A
503,ZION,57.815,N/A,1.203456,N/A,1.092248,N/A,1.233231,N/A,1.048124,N/A


In [24]:
symbol = 'CINF'
tickerData = yf.Ticker(symbol)
ticker_df = tickerData.history(period='1y', end='2025-08-15')
last_price = ticker_df['Close'].iloc[-1]
first_price = ticker_df['Open'].iloc[0]

In [31]:
price_change = float(last_price / first_price)

In [32]:
price_change

1.2069311370498426

In [27]:
hqm_columns = [
    'Ticker',
    'Price',
    'Number of Shares to Buy',
    'One-year Price Return',
    'One-year Return Percentile',
    'Six-month Price Return',
    'Six-month Return Percentile',
    'Three-month Price Return',
    'Three-month Return Percentile',
    'One-month Price Return',
    'One-month Return Percentile',
    
]

hqm_df = pd.DataFrame(columns=hqm_columns)
today = datetime.today().strftime('%Y-%m-%d')


for symbol_string in symbol_strings:
    tickers = yf.Tickers(symbol_string) 
    for symbol in symbol_string.split(' '):
        print(symbol)
        try:
            
            tickerData = yf.Ticker(symbol)
            ticker_df = tickerData.history(period='1y', end=today)
            last_price = ticker_df['Close'].iloc[-1]
            first_price = ticker_df['Open'].iloc[0]
            return_1y = float(last_price / first_price)

            ticker_df = tickerData.history(period='6mo', end=today)
            last_price = ticker_df['Close'].iloc[-1]
            first_price = ticker_df['Open'].iloc[0]
            return_6m = float(last_price / first_price)

            ticker_df = tickerData.history(period='3mo', end=today)
            last_price = ticker_df['Close'].iloc[-1]
            first_price = ticker_df['Open'].iloc[0]
            return_3m = float(last_price / first_price)

            ticker_df = tickerData.history(period='1mo', end=today)
            last_price = ticker_df['Close'].iloc[-1]
            first_price = ticker_df['Open'].iloc[0]
            return_1m = float(last_price / first_price)
            
            price = tickers.tickers[symbol].info['currentPrice']
        except:
            print(f"{symbol} is skipped")
            pass  
            
        series = pd.Series([symbol, price, 'N/A', return_1y, 'N/A', return_6m, 'N/A', return_3m, 'N/A', return_1m, 'N/A'])
        hqm_df = pd.concat([hqm_df, series.to_frame().T], ignore_index=True)
        


A
AAL
AAP
AAPL
ABBV
ABC


$ABC: possibly delisted; no timezone found


ABC is skipped
ABMD


$ABMD: possibly delisted; no timezone found


ABMD is skipped
ABT
ACN
ADBE
ADI
ADM
ADP
ADSK
AEE
AEP
AES
AFL
AIG
AIV
AIZ
AJG
AKAM
ALB
ALGN
ALK
ALL
ALLE
ALXN


...

WLTW is skipped
WM
WMB
WMT
WRB
WRK


$WRK: possibly delisted; no timezone found


WRK is skipped
WST
WU
WY
WYNN
XEL
XLNX


$XLNX: possibly delisted; no timezone found


XLNX is skipped
XOM
XRAY
XRX
XYL
YUM
ZBH
ZBRA
ZION
ZTS


In [30]:
hqm_df.columns = hqm_columns = [
    'Ticker',
    'Price',
    'Number of Shares to Buy',
    'One-year Price Return',
    'One-year Return Percentile',
    'Six-month Price Return',
    'Six-month Return Percentile',
    'Three-month Price Return',
    'Three-month Return Percentile',
    'One-month Price Return',
    'One-month Return Percentile',
    
]

In [31]:
hqm_df

,Ticker,Price,Number of Shares to Buy,One-year Price Return,One-year Return Percentile,Six-month Price Return,Six-month Return Percentile,Three-month Price Return,Three-month Return Percentile,One-month Price Return,One-month Return Percentile
0,A,124.96,N/A,0.886333,N/A,0.978855,N/A,1.070741,N/A,1.043507,N/A
1,AAL,13.25,N/A,1.293945,N/A,0.909403,N/A,1.138316,N/A,1.141258,N/A
2,AAP,60.52,N/A,1.32628,N/A,1.65543,N/A,1.149734,N/A,1.052339,N/A
3,AAPL,232.56,N/A,1.015395,N/A,0.983875,N/A,1.143648,N/A,1.087048,N/A
4,ABBV,207.92,N/A,1.098755,N/A,1.031854,N/A,1.143115,N/A,1.099233,N/A
...,...,...,...,...,...,...,...,...,...,...,...
500,YUM,145.47,N/A,1.08671,N/A,0.948197,N/A,1.015852,N/A,1.006643,N/A
501,ZBH,105.06,N/A,0.922649,N/A,1.007138,N/A,1.140332,N/A,1.098609,N/A
502,ZBRA,322.0,N/A,0.932145,N/A,1.038542,N/A,1.100667,N/A,0.954753,N/A
503,ZION,57.815,N/A,1.203456,N/A,1.092248,N/A,1.233231,N/A,1.048124,N/A


In [13]:
! pwd

/Users/lin/github/quant


In [32]:
hqm_df.to_csv('hqm_df.csv', index=False)

## Calculating Momentum Percentiles

We now need to calculate momentum percentile scores for every stock in the universe. More specifically, we need to calculate percentile scores for the following metrics for every stock:

* `One-Year Price Return`
* `Six-Month Price Return`
* `Three-Month Price Return`
* `One-Month Price Return`

Here's how we'll do this:

In [34]:
hqm_df = pd.read_csv('/Users/lin/github/quant/hqm_df.csv')

In [35]:
time_periods = ['One-year', 'Six-month', 'Three-month', 'One-month']

for row in hqm_df.index:
    for time_period in time_periods:
        return_col = f'{time_period} Price Return'
        percentile_col = f"{time_period} Return Percentile"
        hqm_df.loc[row, percentile_col] = stats.percentileofscore(hqm_df[return_col], hqm_df.loc[row, return_col])
        

In [36]:
hqm_df.sort_values('One-year Return Percentile', ascending=False).head(10)

,Ticker,Price,Number of Shares to Buy,One-year Price Return,One-year Return Percentile,Six-month Price Return,Six-month Return Percentile,Three-month Price Return,Three-month Return Percentile,One-month Price Return,One-month Return Percentile
441,TPR,102.61,NaN,2.580443,100.000000,1.231982,83.366337,1.274615,91.485149,0.947461,16.633663
454,UAL,105.08,NaN,2.524141,99.801980,1.133182,73.267327,1.329453,94.851485,1.132206,92.673267
391,RE,365.84,NaN,2.192333,99.504950,1.543616,98.316832,1.459894,98.514851,1.124451,92.178218
390,RCL,365.84,NaN,2.192333,99.504950,1.543616,98.316832,1.459894,98.514851,1.124451,92.178218
50,AVGO,308.65,NaN,1.969635,99.207921,1.585169,98.613861,1.258088,90.297030,1.037479,63.960396
84,CCL,32.49,NaN,1.935080,99.009901,1.379618,93.663366,1.409545,97.227723,1.100610,89.702970
340,NRG,148.66,NaN,1.875064,98.811881,1.432362,95.247525,0.942772,16.039604,0.934145,12.277228
233,HWM,176.16,NaN,1.824168,98.613861,1.322352,89.900990,1.036250,40.396040,0.923833,9.900990
422,STX,172.38,NaN,1.788782,98.415842,1.742788,99.801980,1.474102,99.009901,1.135947,93.069307
332,NFLX,1231.45,NaN,1.784710,98.217822,1.270112,86.534653,1.019412,34.257426,1.044487,68.712871


## Calculating the HQM Score

We'll now calculate our `HQM Score`, which is the high-quality momentum score that we'll use to filter for stocks in this investing strategy.

The `HQM Score` will be the arithmetic mean of the 4 momentum percentile scores that we calculated in the last section.

To calculate arithmetic mean, we will use the `mean` function from Python's built-in `statistics` module.

In [37]:
for row in hqm_df.index:
    momentum_percentiles = []
    for time_period in time_periods:
        momentum_percentiles.append(hqm_df.loc[row, f'{time_period} Return Percentile'])
    hqm_df.loc[row, 'HQM Score'] = mean(momentum_percentiles)

## Selecting the 50 Best Momentum Stocks

As before, we can identify the 50 best momentum stocks in our universe by sorting the DataFrame on the `HQM Score` column and dropping all but the top 50 entries.

In [39]:
top_50 = hqm_df.sort_values('HQM Score', ascending=False).head(50)

In [44]:
top_50.reset_index(inplace=True)

In [45]:
top_50

,index,Ticker,Price,Number of Shares to Buy,One-year Price Return,One-year Return Percentile,Six-month Price Return,Six-month Return Percentile,Three-month Price Return,Three-month Return Percentile,One-month Price Return,One-month Return Percentile,HQM Score
0,479,WDC,82.04,NaN,1.718684,97.425743,1.700659,99.405941,1.544398,99.603960,1.181282,97.623762,98.514851
1,422,STX,172.38,NaN,1.788782,98.415842,1.742788,99.801980,1.474102,99.009901,1.135947,93.069307,97.574257
2,391,RE,365.84,NaN,2.192333,99.504950,1.543616,98.316832,1.459894,98.514851,1.124451,92.178218,97.128713
3,390,RCL,365.84,NaN,2.192333,99.504950,1.543616,98.316832,1.459894,98.514851,1.124451,92.178218,97.128713
4,37,ANET,136.23,NaN,1.599789,95.643564,1.497527,97.227723,1.501985,99.207921,1.154491,95.445545,96.881188
5,202,GLW,68.93,NaN,1.662501,96.633663,1.422502,94.851485,1.383879,96.831683,1.170885,96.633663,96.237624
6,493,WYNN,126.74,NaN,1.700545,97.029703,1.449532,95.643564,1.376126,96.237624,1.139284,93.663366,95.643564
7,152,EBAY,92.80,NaN,1.627992,96.435644,1.476012,96.435644,1.281412,91.881188,1.173941,96.831683,95.396040
8,331,NEM,72.97,NaN,1.421566,89.306931,1.772761,100.000000,1.378874,96.435644,1.143193,94.059406,94.950495
9,84,CCL,32.49,NaN,1.935080,99.009901,1.379618,93.663366,1.409545,97.227723,1.100610,89.702970,94.900990


## Calculating the Number of Shares to Buy

We'll use the `portfolio_input` function that we created earlier to accept our portfolio size. Then we will use similar logic in a `for` loop to calculate the number of shares to buy for each stock in our investment universe.

## Formatting Our Excel Output

We will be using the XlsxWriter library for Python to create nicely-formatted Excel files.

XlsxWriter is an excellent package and offers tons of customization. However, the tradeoff for this is that the library can seem very complicated to new users. Accordingly, this section will be fairly long because I want to do a good job of explaining how XlsxWriter works.

## Creating the Formats We'll Need For Our .xlsx File

You'll recall from our first project that formats include colors, fonts, and also symbols like % and $. We'll need four main formats for our Excel document:

* String format for tickers
* \$XX.XX format for stock prices
* \$XX,XXX format for market capitalization
* Integer format for the number of shares to purchase

Since we already built our formats in the last section of this course, I've included them below for you. Run this code cell before proceeding.

In [ ]:
background_color = '#0a0a23'
font_color = '#ffffff'

string_template = writer.book.add_format(
        {
            'font_color': font_color,
            'bg_color': background_color,
            'border': 1
        }
    )

dollar_template = writer.book.add_format(
        {
            'num_format':'$0.00',
            'font_color': font_color,
            'bg_color': background_color,
            'border': 1
        }
    )

integer_template = writer.book.add_format(
        {
            'num_format':'0',
            'font_color': font_color,
            'bg_color': background_color,
            'border': 1
        }
    )

percent_template = writer.book.add_format(
        {
            'num_format':'0.0%',
            'font_color': font_color,
            'bg_color': background_color,
            'border': 1
        }
    )

## Saving Our Excel Output

As before, saving our Excel output is very easy: